# IFS HRES &amp; GraphCast Ablation Studies

This notebook runs the ablation and comparison for IFS HRES and GraphCast. It includes:
1. Standard Comparison (allowing ERA5 or IFS as a reference).
2. Surface Pressure Derivation Ablation (HRES vs Hypso vs Ref SP).
3. Hydrostatic Balance temperature method (Tv vs T).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline

BASE_DIR = Path("/home/ekasteleyn/aurora_thesis/neuripspaper")
PLOTS_DIR = BASE_DIR / "plots_notebook"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Toggle this to switch between ERA5 and IFS analysis
REFERENCE_DATASET = "ifs" # Options: "era5" or "ifs"

# Base result directory based on reference
if REFERENCE_DATASET == "ifs":
    RESULTS_DIR = BASE_DIR / "results_ifs"
    HYDRO_DIR = BASE_DIR / "results_ifs_hydro"
    HYPSO_DIR = BASE_DIR / "results_ifs_hypso"
    REFSP_DIR = BASE_DIR / "results_ifs_refsp"
else:
    RESULTS_DIR = BASE_DIR / "results"
    HYDRO_DIR = BASE_DIR / "results_hydro" # Assuming similar naming conventions
    HYPSO_DIR = BASE_DIR / "results_hypso"
    REFSP_DIR = BASE_DIR / "results_refsp"

MODELS = ["hres", "graphcast"]
NICE = {"hres": "HRES", "graphcast": "GraphCast"}
MODEL_STYLES = {
    "graphcast": {"color": "#000000", "marker": "D"},
    "hres":      {"color": "#56B4E9", "marker": "P"},
    "ifs_ablation_hypso": {"color": "#D55E00", "marker": "s"},
    "ifs_ablation_refsp": {"color": "#009E73", "marker": "^"}
}

## 1. General Comparison (HRES vs GraphCast)
Comparison of Conservation and Balance properties. 

In [ ]:
def plot_combined_conservation(results_dir, models, outname):
    frames = []
    for m in models:
        for path in results_dir.glob(f"time_series_{m}_*.csv"):
            df = pd.read_csv(path)
            df["model"] = m
            frames.append(df)
            
    if not frames: 
        print("No timeseries data found.")
        return
    df_all = pd.concat(frames, ignore_index=True)

    metrics = [("dry_mass_Eg", "Dry Air Mass (Eg)"),
               ("water_mass_kg", "Water Mass (kg)"),
               ("total_energy_J", "Total Energy (J)")]
               
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    
    for ax, (col, title) in zip(axes, metrics):
        for model in models:
            mdf = df_all[df_all["model"] == model]
            if mdf.empty or col not in mdf.columns: continue
            
            rel_df = mdf[["date", "forecast_hour", col]].dropna().copy()
            if rel_df.empty: continue
            base = rel_df.sort_values("forecast_hour").groupby("date", as_index=False).first()[["date", col]].rename(columns={col: "base_val"})
            rel_df = rel_df.merge(base, on="date", how="left")
            rel_df = rel_df[rel_df["base_val"].abs() > 0]
            if rel_df.empty: continue
            
            rel_df["rel_pct"] = (rel_df[col] - rel_df["base_val"]) / rel_df["base_val"] * 100.0
            agg = rel_df.groupby("forecast_hour")["rel_pct"].agg(["mean", "std"]).reset_index()
            
            style = MODEL_STYLES.get(model, {"color": "grey", "marker": "."})
            ax.plot(agg["forecast_hour"], agg["mean"], label=NICE.get(model, model), color=style["color"], marker=style["marker"], markersize=4)
            ax.fill_between(agg["forecast_hour"], agg["mean"] - agg["std"].fillna(0), agg["mean"] + agg["std"].fillna(0), color=style["color"], alpha=0.18, linewidth=0)
            
        ax.set_title(title, fontsize=16)
        ax.set_xlabel("Forecast Hour", fontsize=14)
        if ax == axes[0]: ax.set_ylabel("Relative Change (%)", fontsize=14)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=len(labels), fontsize=14)
    fig.suptitle(f"Conservation comparison ({REFERENCE_DATASET.upper()} Reference)", fontsize=18)
    fig.tight_layout()
    plt.savefig(PLOTS_DIR / outname, bbox_inches="tight")
    plt.show()

print("General Conservation:")
plot_combined_conservation(RESULTS_DIR, MODELS, "hres_gc_conservation.png")

## 2. Surface Pressure Derivation Ablation
Comparing HRES standard, Hypso, and Reference Surface Pressure methods.

In [ ]:
ablation_models = ["hres", "ifs_ablation_hypso", "ifs_ablation_refsp"]
NICE.update({
    "ifs_ablation_hypso": "HRES (Hypso)",
    "ifs_ablation_refsp": "HRES (Ref SP)"
})

frames_sp = []
for m in ablation_models:
    # Determine which folder to load for each model
    if m == "hres": d = RESULTS_DIR
    elif m == "ifs_ablation_hypso": d = HYPSO_DIR
    elif m == "ifs_ablation_refsp": d = REFSP_DIR
    else: continue
    
    for path in d.glob(f"time_series_{m}_*.csv"):
        df = pd.read_csv(path)
        df["model"] = m
        frames_sp.append(df)

if frames_sp:
    df_sp_all = pd.concat(frames_sp, ignore_index=True)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    metrics = [("dry_mass_Eg", "Dry Air Mass"), ("water_mass_kg", "Water Mass"), ("total_energy_J", "Total Energy")]

    for ax, (col, title) in zip(axes, metrics):
        for model in ablation_models:
            mdf = df_sp_all[df_sp_all["model"] == model]
            if mdf.empty or col not in mdf.columns: continue
            
            rel_df = mdf[["date", "forecast_hour", col]].dropna().copy()
            if rel_df.empty: continue
            base = rel_df.sort_values("forecast_hour").groupby("date", as_index=False).first()[["date", col]].rename(columns={col: "base_val"})
            rel_df = rel_df.merge(base, on="date", how="left")
            rel_df = rel_df[rel_df["base_val"].abs() > 0]
            if rel_df.empty: continue
            
            rel_df["rel_pct"] = (rel_df[col] - rel_df["base_val"]) / rel_df["base_val"] * 100.0
            agg = rel_df.groupby("forecast_hour")["rel_pct"].agg(["mean", "std"]).reset_index()
            style = MODEL_STYLES.get(model, {"color": "grey", "marker": "."})
            ax.plot(agg["forecast_hour"], agg["mean"], label=NICE.get(model, model), color=style["color"], marker=style["marker"], markersize=4)

        ax.set_title(title, fontsize=16)
        ax.set_xlabel("Forecast Hour", fontsize=14)

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=len(labels), fontsize=14)
    fig.suptitle("Surface Pressure Derivation Ablation", fontsize=18)
    fig.tight_layout()
    plt.savefig(PLOTS_DIR / "sp_ablation.png", bbox_inches="tight")
    plt.show()
else:
    print("No SP ablation data found.")

## 3. Hydrostatic Temperature Ablation (Tv vs T)
Comparing standard Virtual Temperature (Tv) vs regular Temperature (T) formulation.

In [ ]:
variants = {"Tv": RESULTS_DIR, "T": HYDRO_DIR}

hydro_frames = []
for v_name, v_dir in variants.items():
    if not v_dir.exists(): continue
    for m in MODELS:
        for path in v_dir.glob(f"time_series_{m}_*.csv"):
            df = pd.read_csv(path)
            if "hydrostatic_rmse" in df.columns:
                df["variant"] = v_name
                df["model"] = m
                hydro_frames.append(df)

if hydro_frames:
    df_hydro = pd.concat(hydro_frames, ignore_index=True)
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for model in MODELS:
        for v_name in variants.keys():
            mdf = df_hydro[(df_hydro["model"] == model) & (df_hydro["variant"] == v_name)]
            if mdf.empty: continue
            
            # Simple baseline subtraction can be inserted here if baselines are known
            agg = mdf.groupby("forecast_hour")["hydrostatic_rmse"].agg(["mean", "std"]).reset_index()
            
            style = MODEL_STYLES.get(model, {"color": "grey", "marker": "."})
            ls = "-" if v_name == "Tv" else "--"
            label = f"{NICE.get(model, model)} ({v_name})"
            
            ax.plot(agg["forecast_hour"], agg["mean"], label=label, color=style["color"], marker=style["marker"], markersize=5, linestyle=ls)
            
    ax.set_title("Hydrostatic Balance Comparison (Tv vs T)", fontsize=18)
    ax.set_xlabel("Forecast Hour", fontsize=14)
    ax.set_ylabel("Raw RMSE (m²/s²)", fontsize=14)
    ax.legend(fontsize=12, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    plt.savefig(PLOTS_DIR / "hydro_temperature_ablation.png", bbox_inches="tight")
    plt.show()
else:
    print("No hydrostatic ablation data found.")